# 01 - Ingesta de viajes: descompresión y capa bronze

**Antes de este notebook.** Databricks no llega a los servidores de la IM, así que los archivos se suben desde una computadora local con `python scripts/subir_viajes.py`. Quedan en el volume como `viajes_zip/viajes_stm_AAAA_MM.zip`.

**Parte A. Descompresión.** Extrae cada ZIP del volume a `viajes/viajes_stm_AAAA_MM.csv`.

**Parte B. Bronze.** Convierte cada mes a la tabla Delta `viajes_bronze`, sin limpiar ni transformar los datos: todas las columnas se leen como texto y se agregan columnas de trazabilidad.

Ambas partes son **idempotentes**: los meses ya descomprimidos o ya cargados se saltean, salvo que se fuerce el reproceso.

In [0]:
%run ./00_setup

## Parámetros

| Parámetro | Valor | Efecto |
|---|---|---|
| `MESES` | `["2024-01"]` | Procesa solo esos meses. **Usar un mes en la primera ejecución.** |
| `MESES` | `None` | Procesa todos los meses subidos al volume. |
| `DESCOMPRIMIR` | `False` | Saltea la Parte A (si los CSV ya están en el volume). |
| `FORZAR_BRONZE` | `True` | Vuelve a cargar en bronze meses que ya estaban cargados. |

In [0]:
MESES = None             # todos los meses      
FUENTE = "zip"           # "zip", "volume" o "ckan"
CARPETA_ZIP = config.RAW_ZIP   # ajustar si los ZIP están en otra carpeta del volume
FORZAR_BRONZE = False
 
assert FUENTE in ("zip", "volume", "ckan"), "FUENTE debe ser 'zip', 'volume' o 'ckan'"
DESCARGAR = FUENTE == "ckan"
CARPETA_CSV = config.RAW_VIAJES

In [0]:
import importlib
from pathlib import Path
 
from pyspark.sql import functions as F
 
# Requiere haber ejecutado antes la celda %run ./00_setup (agrega src/ al sys.path)
from bondis import ingesta
 
importlib.reload(ingesta)
 
# descarga.py solo se necesita para FUENTE = "ckan"
if FUENTE == "ckan":
    from bondis import descarga
    importlib.reload(descarga)
 
 
def mes_elegido(anio_mes):
    """True si el mes 'AAAA-MM' entra en el filtro MESES."""
    return MESES is None or anio_mes in MESES

## Parte A0. Descompresión de los ZIP descargados a mano

Para cada ZIP de `CARPETA_ZIP`:
1. Identifica el mes por el nombre del ZIP (o, si no se puede, por el del CSV interno).
2. Extrae el CSV a `raw/viajes/viajes_stm_AAAA_MM.csv`.
3. Verifica que el tamaño extraído coincida con el declarado en el ZIP.

Los ZIP no se borran. Los CSV ya extraídos se saltean.
**Revisar la tabla de meses identificados antes de seguir.**

In [0]:
if FUENTE == "zip":
    # Busca también en subcarpetas y sin distinguir mayúsculas (.zip, .ZIP, .Zip)
    zips = sorted(p for p in Path(CARPETA_ZIP).rglob("*") if p.is_file() and p.suffix.lower() == ".zip")
    print(f"ZIP encontrados en {CARPETA_ZIP}: {len(zips)}\n")
 
    # Vista previa: qué mes se le asigna a cada ZIP, sin extraer nada.
    # Si el nombre del ZIP no tiene el mes, se lee el nombre del CSV interno.
    mes_por_zip = {}
    print(f"{'ZIP':<34} {'Mes':<8} {'ZIP':>8} {'CSV':>8}  CSV interno")
    for z in sorted(zips, key=lambda p: ingesta.mes_de_zip(p) or "9999"):
        am = ingesta.mes_de_zip(z)
        mes_por_zip[z] = am
        internos = ingesta.contenido_zip(z)
        nombre_csv = internos[0][0] if internos else "(sin CSV)"
        mb_csv = sum(mb for _, mb in internos)
        print(f"{z.name:<34} {am or '¿?':<8} {z.stat().st_size / 1e6:>6,.0f}MB {mb_csv:>6,.0f}MB  {nombre_csv}")
 
    # Control de cobertura: meses repetidos y meses faltantes del período del proyecto
    from collections import Counter
    conteo = Counter(am for am in mes_por_zip.values() if am)
    repetidos = sorted(am for am, n in conteo.items() if n > 1)
    esperados = (
        spark.sql(f"SELECT explode(sequence(to_date('{config.FECHA_INICIO}'), "
                  f"date_sub(to_date('{config.FECHA_FIN}'), 1), interval 1 month)) AS d")
        .selectExpr("date_format(d, 'yyyy-MM') AS am").toPandas()["am"].tolist()
    )
    faltantes = [am for am in esperados if am not in conteo]
    print(f"\nMeses identificados: {len(conteo)}  ·  sin identificar: {sum(1 for a in mes_por_zip.values() if not a)}")
    print(f"Meses repetidos: {repetidos or 'ninguno'}")
    print(f"Meses del período sin ZIP: {faltantes or 'ninguno'}")

In [0]:
errores_zip = []
 
if FUENTE == "zip":
    for z in zips:
        am = ingesta.mes_desde_texto(z.stem)
        if am is not None and not mes_elegido(am):
            continue
        try:
            am, _ = ingesta.descomprimir_zip(z, config.RAW_VIAJES)
        except Exception as e:  # se registra y se sigue con el próximo ZIP
            errores_zip.append((z.name, str(e)))
            print(f"[FAIL] {z.name}: {e}")
 
    print(f"\nErrores: {len(errores_zip)}")
    for nombre, err in errores_zip:
        print(f"  {nombre}: {err}")

## Parte B. Carga a bronze
### B.1 Inventario de archivos y estado de la tabla

In [0]:
archivos = sorted(Path(CARPETA_CSV).glob("viajes_stm_*.csv"))
print(f"Carpeta de origen: {CARPETA_CSV}  ({len(archivos)} CSV de viajes en total)")
por_mes = {am: ps for am, ps in ingesta.agrupar_por_mes(archivos).items() if mes_elegido(am)}
 
tabla_existe = spark.catalog.tableExists(config.TABLE_BRONZE)
cargados = set()
if tabla_existe:
    cargados = {
        r["anio_mes"]
        for r in spark.table(config.TABLE_BRONZE).select("anio_mes").distinct().collect()
    }
 
pendientes = {am: ps for am, ps in por_mes.items() if FORZAR_BRONZE or am not in cargados}
 
print(f"Meses con CSV en el volume: {len(por_mes)}")
print(f"Meses ya cargados en bronze: {len(cargados & set(por_mes))}")
print(f"Meses a cargar ahora:        {len(pendientes)}  {list(pendientes)}")

### B.2 Carga mes a mes

Para cada mes:
1. Detecta separador y codificación mirando el inicio del archivo.
2. Lee el CSV con **todas las columnas como texto** (bronze no interpreta tipos).
3. Normaliza los nombres de columnas.
4. Agrega `anio_mes`, `archivo_origen` y `fecha_ingesta`.
5. Reemplaza **solo la partición de ese mes** en la tabla (`replaceWhere`).

In [0]:
columnas_por_mes = {}
 
for am, paths in pendientes.items():
    sep, enc, encabezado = ingesta.detectar_formato(paths[0])
    print(f"[{am}] {len(paths)} archivo/s · separador {sep!r} · {enc}")
 
    crudo = (
        spark.read
        .option("header", True)
        .option("sep", sep)
        .option("encoding", enc)
        .option("quote", '"')
        .option("escape", '"')
        .option("inferSchema", False)
        .csv(paths)
    )
 
    nuevos_nombres = ingesta.normalizar_columnas(crudo.columns)
    df = crudo.select(
        *[F.col(f"`{c}`").alias(n) for c, n in zip(crudo.columns, nuevos_nombres)],
        F.lit(am).alias("anio_mes"),
        F.col("_metadata.file_name").alias("archivo_origen"),
        F.current_timestamp().alias("fecha_ingesta"),
    )
 
    escritor = df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
    if tabla_existe:
        escritor.option("replaceWhere", f"anio_mes = '{am}'").saveAsTable(config.TABLE_BRONZE)
    else:
        escritor.partitionBy("anio_mes").saveAsTable(config.TABLE_BRONZE)
        tabla_existe = True
 
    filas = spark.table(config.TABLE_BRONZE).where(F.col("anio_mes") == am).count()
    columnas_por_mes[am] = nuevos_nombres
    print(f"        {filas:,} filas · columnas: {nuevos_nombres}")

### B.3 Control de consistencia de columnas

Si la IM cambió el formato entre años, distintos meses tendrán columnas distintas.
Bronze las conserva todas (`mergeSchema`); la unificación se resuelve en silver.

In [0]:
if columnas_por_mes:
    esquemas = {}
    for am, cols in columnas_por_mes.items():
        esquemas.setdefault(tuple(cols), []).append(am)
    if len(esquemas) == 1:
        print("Todos los meses cargados tienen las mismas columnas.")
    else:
        print(f"ATENCIÓN: {len(esquemas)} esquemas distintos.")
        for cols, meses in esquemas.items():
            print(f"  Meses {meses}:\n    {list(cols)}")
else:
    print("No se cargaron meses en esta ejecución.")

## Verificación

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    display(
        spark.table(config.TABLE_BRONZE)
        .groupBy("anio_mes")
        .agg(F.count("*").alias("filas"), F.countDistinct("archivo_origen").alias("archivos"))
        .orderBy("anio_mes")
    )
 
# COMMAND ----------
 
if spark.catalog.tableExists(config.TABLE_BRONZE):
    spark.table(config.TABLE_BRONZE).printSchema()
    display(spark.table(config.TABLE_BRONZE).limit(10))

In [0]:
if spark.catalog.tableExists(config.TABLE_BRONZE):
    spark.table(config.TABLE_BRONZE).printSchema()
    display(spark.table(config.TABLE_BRONZE).limit(10))